## Imports

In [50]:
import numpy as np
import pandas as pd

import math
import random
import gc

## Loading
Loading the data, perhaps via chunking, but this may cause issues with duplicate detection later on

In [51]:
"""
Loads chunks of data from csv file

Returns:
    pd.DataFrame: The selected chunk of data
"""
def load_csv_chunk(
    filepath: str,              # Path to .csv file
    chunks: int = 1,            # Number of (roughly) equal parts to split the csv into
    chunk_idx: int = 0,         # Index of the chunk to load (zero-based)
    offset: int = 1,            # Number of header lines in file
    **read_csv_kwargs           # Extra arguments passed to pd.read_csv()
):

    # Count number of data rows, excluding the header
    #   Get specified encoding, otherwise default utf-8
    with open(filepath, "r", encoding=read_csv_kwargs.get("encoding", "utf-8")) as f:
        total_rows = sum(1 for line in f) - offset

    # Chunking/Slicing Information
    chunk_size = math.ceil(total_rows / chunks)           # Size of chunk
    start_row = chunk_idx * chunk_size                     # Index of start row in actual data
    end_row = min(start_row + chunk_size, total_rows)     # Index of end row in actual data
    nrows = end_row - start_row                           # Number of rows to read
    header_row = offset - 1                               # Row index for the header information (assumed as the last header row)

    # Extract the header/columns
    columns = pd.read_csv(
        filepath,
        skiprows=header_row,
        nrows=0,
        **read_csv_kwargs
    ).columns

    # Read only the selected chunk of data
    df = pd.read_csv(
        filepath,
        skiprows=offset + start_row,
        nrows=nrows,
        names=columns,
        header=None,
        **read_csv_kwargs
    )

    return df

In [52]:
raw_sample = load_csv_chunk("UK-Sanctions-List.csv", offset=2)

C:\Users\alecz\AppData\Local\Temp\ipykernel_26764\4224360936.py:36: DtypeWarning: Columns (0: IMO number, 1: Current owner/operator (s), 2: Previous owner/operator (s), 3: Current believed flag of ship, 4: Previous flags, 5: Type of ship) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


## Exploration
Explore the fields (columns), look for data we're interested in (Name, DOB, Countries, other...)

By the end of this step, we should have an understanding of which fields are useful and which can be excluded.

In [133]:
"""
Gives useful information and examples of data in a specific column
Helps decide if it is useful for our purposes
"""
def profile_column(
        df: pd.DataFrame,                # Dataframe to analyse
        col_name: str,                   # Column identifier
        top_n: int = 5,                  # Show top_n most common values
        sampl_n: int = 10,               # Show sampl_n number of random examples
        key: list[str] | None = None     # Print Key fields to the examples
) -> None:
    
    if key is None:
        key = []
    
    col = df[col_name]
    rows = len(col)
    missg = col.isna().sum()
    non_missg = col.notna().sum()
    missg_pct = round((missg / rows) * 100, 2)
    unique = col.nunique(dropna=True)

    # Basic Info
    print("#" * 40)
    print(f"Column: {col_name}")
    print("#" * 40)
    print(f"Total rows:          {rows}")
    print(f"Data type:           {col.dtype}")
    print(f"Non-missing values:  {non_missg}")
    print(f"Missing values:      {missg}")
    print(f"Missing %:           {missg_pct}%")
    print(f"Unique values:       {unique}")
    if non_missg > 0:
        print(f"Most common value:   {col.value_counts(dropna=True).index[0]}")
        print(f"Most common count:   {col.value_counts(dropna=True).iloc[0]}")

    # Print top occurences
    print("\nTop value counts:")
    print("-" * 20)
    print(col.value_counts(dropna=False).head(top_n))


    # Build Example Data
    example_cols = key + [col_name]
    examples = (
        df[df[col_name].notna()]
        [example_cols]
        .drop_duplicates(subset=[col_name])   # drops duplicates across col_name
    )


    # Print some random examples
    print("\nRandom non-missing examples:")
    print("-" * 20)
    if len(examples) == 0:
        print("No non-missing examples available.")
    else:
        print(
            examples
            .sample(min(sampl_n, len(examples)))
            .to_string(index=False, header=False)
        )


    # Print longest example (if text area)
    print("\nLongest non-missing examples:")
    print("-" * 20)
    if len(examples) == 0:
        print("No non-missing examples available.")
    else:
        longest_examples = (
            examples
            .assign(length=examples[col_name].astype(str).str.len())   # creates new length column for sorting
            .sort_values("length", ascending=False)
            .drop(columns="length")                                    # drop length column on display                   
            .head(sampl_n)
        )
        print(longest_examples.to_string(index=False, header=False))

    
    # Print shortest example (sneaky NaNs)
    print("\nShortest non-missing examples:")
    print("-" * 20)
    if len(examples) == 0:
        print("No non-missing examples available.")
    else:
        shortest_examples = (
            examples
            .assign(length=examples[col_name].astype(str).str.len())
            .sort_values("length", ascending=True)
            .drop(columns="length")
            .head(sampl_n)
        )
        print(shortest_examples.to_string(index=False, header=False))


In [54]:
# Other Basic Tools
# df.shape, df.head()
# df.columns
# df.dtypes, df.info()
# df.isna().sum(), df.count().sort_values(ascending=True)
# df.nunique(), df["some_column"].value_counts(dropna=False)

In [55]:
raw_sample.head()

,Last Updated,Unique ID,OFSI Group ID,UN Reference Number,Name 6,Name 1,Name 2,Name 3,Name 4,Name 5,...,IMO number,Current owner/operator (s),Previous owner/operator (s),Current believed flag of ship,Previous flags,Type of ship,Tonnage of ship,Length of ship,Year Built,Hull identification number (HIN)
0,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,16/04/2026,AFG0001,12703.0,TAe.010,HAJI KHAIRULLAH HAJI SATTAR MONEY EXCHANGE,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
raw_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 57033 entries, 0 to 57032
Data columns (total 58 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Last Updated                                57033 non-null  str    
 1   Unique ID                                   57033 non-null  str    
 2   OFSI Group ID                               54769 non-null  float64
 3   UN Reference Number                         20532 non-null  str    
 4   Name 6                                      56921 non-null  str    
 5   Name 1                                      23893 non-null  str    
 6   Name 2                                      12107 non-null  str    
 7   Name 3                                      2573 non-null   str    
 8   Name 4                                      477 non-null    str    
 9   Name 5                                      101 non-null    str    
 10  Name type            

In [57]:
profile_column(raw_sample, "Subsidiaries", top_n=10)

########################################
Column: Subsidiaries
########################################
Total rows:          57033
Data type:           str
Non-missing values:  5595
Missing values:      51438
Missing %:           90.19%
Unique values:       206
Most common value:   AIS Iran Co
Most common count:   420

Top value counts:
--------------------
Subsidiaries
NaN                                                51438
AIS Iran Co                                          420
Electronic Component Industries (ECI)                420
Iranian Electronic Science & Research Institute      420
Iran Electronics Industries Co (Saga)                420
Isfahan Optics Industry (SAPA)                       420
Security Industry Information Space (SASTOBA)        420
Shiraz Electronics Industries (Sara Shiraz)          420
Telecommunication Industries of Iran (SAMA)          420
The Institute of Isayran Co                          420
Name: count, dtype: int64

Random non-missing examples:
--

In [58]:
"""
Other experiments
"""

# Checking pipelining in Subsidiaries and Parent company
# filt = raw_sample[
#     (raw_sample["Subsidiaries"].str.contains(";"))
# ]
# filt["Subsidiaries"]


# Checking entity specific information, ie. ships don't have passport number etc...
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Phone number", "Website", 
#               "Email address", "National Identifier number", 
#               "Passport number", "Business registration number (s)", "IMO number"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Checking only individuals have DOB and Gender
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["D.O.B", "Gender"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Checking Country information per entity type
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Nationality(/ies)", "Country of birth", "Current believed flag of ship", "Previous flags"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")
    
# Making sure only entities have subsidiaries or parent companies
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Subsidiaries", "Parent company"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")

# Making sure only entities have subsidiaries or parent companies
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Title"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# Seeing name relationships across entities (drop these later for space)
# for ent in ["Individual", "Entity", "Ship"]:
#     for f in ["Name type", "Alias strength", "Name non-latin script", "Non-latin script type", "Non-latin script language"]:
#         filt = raw_sample[
#             (raw_sample["Designation Type"] == ent) &
#             (raw_sample[f].notna())
#         ]
#         print(f"{ent}-{f}-found: {len(filt)}")


# filt = raw_sample[
#     (raw_sample["Designation Type"] == "Ship") &
#     (raw_sample["Name 6"].notna())
# ]
# filt


# Check the relationship between designation type and type of entity (does it apply to businesses only?)
# filt = raw_sample[
#     (raw_sample["Designation Type"] == "Entity") &
#     (raw_sample["Type of entity"].notna())
# ]
# len(filt)



# Seeing how many NaNs for alias strength there are even if the name type is an alias
# filt = raw_sample[
#     ((raw_sample["Name type"] == "Alias") | (raw_sample["Name type"] == "ALias")) &
#     (raw_sample["Alias strength"].notna())
# ]
# len(filt)


# filt = raw_sample[
#     (raw_sample["Name type"].isna())
# ]
# filt.iloc[0]


# Date Designated (most recent)
# pd.to_datetime(raw_sample["Date Designated"]).max()


# Address Format checking
# filt = raw_sample[
#     (raw_sample["Address Line 1"].str.contains("TSUEN WAN, NEW TERRITORIES, HONG KONG,UNIT 601", na=False))
# ]
# filt.iloc[0]

# Country Info Checking
# filt = raw_sample[
#     (raw_sample["Town of birth"].notna()) &
#     (raw_sample["Country of birth"].isna()) &
#     (raw_sample["Address Country"].isna()) &
#     (raw_sample["Nationality(/ies)"].isna())
# ]

# ID Differentiation
# filt = raw_sample[
#     (raw_sample["UN Reference Number"] == "TAi.004") 
# ]
# filt["Unique ID"].value_counts()

# Spotting Associations across fields
# filt[["Regime Name", "Designation Type", "Designation source", "Address Country", "Nationality(/ies)", "Country of birth"]].sample(n=20, replace=True)# ["Regime Name"].iloc[0]

# Non-latin script type and Non-latin script language
# filt = raw_sample[
#     (raw_sample["Name non-latin script"].notna()) &
#     (raw_sample["Non-latin script type"].isna()) 
# ]
# filt.iloc[0]


# Checking Name poor formatting
# filt = raw_sample[
#     (raw_sample["Name non-latin script"].notna()) &
#     (raw_sample["Name 1"].isna()) &
#     (raw_sample["Name 6"].isna())
# ]
# filt.iloc[0]


'\nOther experiments\n'

### Findings

Guiding the data cleaning process.Metadata found here: https://www.gov.uk/guidance/format-guide-for-the-uk-sanctions-list

The relevant columns that will contribute to building the transformed data source directly:
   - <span style="color: #61E283;">**Unique ID**</span>: Leaving this in as a reference/foreign key to the original sanctions dataset could help validate their presence in the original dataset later on. Not useful for matching, but has no missing values and could help keep a relationship with our transformed dataset.
   - <span style="color: #61E283;">**Name 1-6**</span>: Very important name information. Name 6 is the surname or the full name of the entity/ship. Names 1-5 are first and middle names, with increasing missing values as expected for rarer longer names
   - <span style="color: #61E283;">**Name type**</span>: Potentially helpful information if aliases are matched with innocent parties, this can be used with alias strength to determine a matching confidence score
   - <span style="color: #61E283;">**Alias strength**</span>: as above
   - <span style="color: #61E283;">**Name non-latin script**</span>: certainly useful for those few percent that may write their names in a different alphabet, helps make the matching system more robust
   - <span style="color: #61E283;">**Regime Name**</span>: not useful for matching but gives the bank context on which sanctions regime applies to the entity, it is also dense with no missing values
   - <span style="color: #61E283;">**Designation Type**</span>: provides context about the name, are they an individual, entity or ship (not directly useful for matching but can contribute to match validation/confidence)
   - <span style="color: #61E283;">**Sanctions Imposed**</span>: provides context about the type of sanctions the matched entity may face (again, not useful directly for matching, but may guide the bank's procedure after matching)
   - <span style="color: #61E283;">**Other Information**</span>: provides context about the person and their sanctions which isn't useful directly for matching, but can help inform later decisions. It may also be used to determine associated countries
   - <span style="color: #61E283;">**UK Statement of Reasons**</span>: as above, provides more context that could be used to further clarify situation after matching, not necessarily useful for the actual matching
   - <span style="color: #61E283;">**Type of entity**</span>: as above, provides more context that could be used to further clarify situation after matching, not necessarily useful for the actual matching (but for business/entity organisations rather than individuals)
   - <span style="color: #61E283;">**Address 1-6, Postal Code, Country**</span>: could use the address to match customer records or further improve matching confidence score. (also find associated countries)
   - <span style="color: #61E283;">**Phone number, Website, Email address, National Identifier number, Passport number, Business registration number, IMO number**</span>: further information that can guide matching confidence or be used as secondary matching criteria
   - <span style="color: #61E283;">**D.O.B, Gender**</span>: futher validation for name matching (eg. two people with same name but different birthdays, one is innocent)
   - <span style="color: #61E283;">**Nationality(/ies), Country of birth, Current believed flag of ship, previous flags**</span>: useful associated countries information
   - <span style="color: #61E283;">**Subsidiaries, Parent Company**</span>: the matching system likely needs to flag these too, treated as separate entity names for example (eg. companies under parent company being sanctioned)
   - <span style="color: #61E283;">**Current owner, Previous owner**</span>: similar to above but for ships
   - <span style="color: #61E283;">**Type of ship, Tonnage of ship, Length, Year built**</span>: validate match confidence but for ships



The columns we decided have no use for us are:
   - <span style="color: #F76262;">**Last Updated**</span>: If an entity already appears in this list, we would want the system to match them. There is no date of release of sanction that could be used alongside this field to clarify the sanctions validity.
   - <span style="color: #F76262;">**Date Designated**</span>: for similar reasons to the above
   - <span style="color: #F76262;">**OFSI Group ID**</span>: It was thought this could be used as a compound key with the Unique ID, but after exploring further, it held no differentiating power (legacy ID)
   - <span style="color: #F76262;">**UN Reference Number**</span>: similar to the above, it was mostly missing and held no differentiating power
   - <span style="color: #F76262;">**Designation source**</span>: it is already assumed we are curating a dataset for UK sanctions, and this field only differentiates between UN or UK which both matter (redundant)
   - <span style="color: #F76262;">**HIN**</span>: NaN column, can be discarded
   

These fields may be helpful, for example for imputation, but will ultimately not be needed in our transformed dataset:
   - <span style="color: #d48748;">**Title**</span>: some titles can be very distinguishing (Second Vice-President of the National Consti...), others may be very general (Captain, General...), shouldn't reliably be used for matching (or even match checking), titles may change, its mostly missing values, but maybe can impute country information or things like that??
   - <span style="color: #d48748;">**Position**</span>: similar to above
   - <span style="color: #d48748;">**Non-latin script type**</span>: not helpful for name matching purposes, but may help discern country information??
   - <span style="color: #d48748;">**Non-latin script language**</span>: as above
   - <span style="color: #d48748;">**National Identifier additional information, Passport additional information**</span>: not useful for matching, does provide context but not highly relevant. it may be used for identifying associated countries from the text though.
   - <span style="color: #d48748;">**Town of birth**</span>: can be used to impute country association data

## Designing
This is where we can start to formulate what our database may look like, consolidating our exploration ideas before data cleaning/transformation ensues

The proposed solution involves cleaning and normalising the dataset into 4 component datasets:
1. <span style="color: #61E283;">**name_index.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: newly created Sanctioned Entity ID, this can function as our new primary key, and foreign key across the datasets
    - <span style="color: #5d8eb8;">Full Name</span>: full name field, including any middle names and surname
    - <span style="color: #5d8eb8;">Designation Type</span>: identifies whether the party is an individual, entity or a ship
2. <span style="color: #61E283;">**individuals.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: Legacy Unique ID, keeps the relationship to the original gov.uk dataset
    - <span style="color: #5d8eb8;">Surname</span>: from Name 6
    - <span style="color: #5d8eb8;">Given Names</span>: Name 1+2+...+5
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">D.O.B</span>: =
    - <span style="color: #5d8eb8;">Gender</span>: =
    - <span style="color: #5d8eb8;">Title</span>: =
    - <span style="color: #5d8eb8;">Position</span>: =
    - <span style="color: #5d8eb8;">Nationalities</span>: =
    - <span style="color: #5d8eb8;">Birth Country</span>: =
    - <span style="color: #5d8eb8;">Birth Town</span>: =
    - <span style="color: #5d8eb8;">Address Lines</span>: Address Lines 1+2+...+6
    - <span style="color: #5d8eb8;">Address Postal Code</span>: =
    - <span style="color: #5d8eb8;">Address Country</span>: =
    - <span style="color: #5d8eb8;">Phone Number</span>: =
    - <span style="color: #5d8eb8;">Website</span>: =
    - <span style="color: #5d8eb8;">Email</span>: =
    - <span style="color: #5d8eb8;">National Identifier Number</span>: =
    - <span style="color: #5d8eb8;">National Identifier Info</span>: =
    - <span style="color: #5d8eb8;">Passport Number</span>: =
    - <span style="color: #5d8eb8;">Passport Info</span>: =

3. <span style="color: #61E283;">**entities.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: ^^
    - <span style="color: #5d8eb8;">Name</span>: Usually just Name 6, but concat 1-6
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">Address Lines</span>: Address Lines 1+2+...+6
    - <span style="color: #5d8eb8;">Address Postal Code</span>: =
    - <span style="color: #5d8eb8;">Address Country</span>: =
    - <span style="color: #5d8eb8;">Phone Number</span>: =
    - <span style="color: #5d8eb8;">Website</span>: =
    - <span style="color: #5d8eb8;">Email</span>: =
    - <span style="color: #5d8eb8;">Business Reg</span>: =
    - <span style="color: #5d8eb8;">Type</span>: from type of entity
    - <span style="color: #5d8eb8;">Subsidiaries</span>: =
    - <span style="color: #5d8eb8;">Parent Company</span>: =
    
4. <span style="color: #61E283;">**ships.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">LUID</span>: ^^
    - <span style="color: #5d8eb8;">Name</span>: Usually just Name 6, but concat 1-6
    - <span style="color: #5d8eb8;">Name (Non-Latin Script)</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Type</span>: =
    - <span style="color: #5d8eb8;">Non-Latin Script Language</span>: =
    - <span style="color: #5d8eb8;">Name Type</span>: =
    - <span style="color: #5d8eb8;">Alias Strength</span>: =
    - <span style="color: #5d8eb8;">IMO Number</span>: =
    - <span style="color: #5d8eb8;">Current Believed Flag</span>: =
    - <span style="color: #5d8eb8;">Previous Flags</span>: =
    - <span style="color: #5d8eb8;">Type</span>: from type of ship
    - <span style="color: #5d8eb8;">Tonnage</span>: =
    - <span style="color: #5d8eb8;">Length</span>: =
    - <span style="color: #5d8eb8;">Year Built</span>: =


5. <span style="color: #61E283;">**sanctions.csv**</span>
    - <span style="color: #5d8eb8;">SEID</span>: ^^
    - <span style="color: #5d8eb8;">Regime Name</span>: =
    - <span style="color: #5d8eb8;">Sanctions Imposed</span>: depipeline this ideally
    - <span style="color: #5d8eb8;">Other Info</span>: =
    - <span style="color: #5d8eb8;">UK Statement of Reasons</span>: =
    - <span style="color: #5d8eb8;">Date Designated</span>: =
    - <span style="color: #5d8eb8;">Last Updated</span>: =

The motivating idea behind this scheme is that it is very likely that the customer records will include a name or a designation type. Here is the typical algorithm flow:

1. Designation Type is known
    - Perform the matching algorithm on the narrower datasets (either: individuals, entities or ships .csv) with whatever fields you have

2. Otherwise, Name is known
    - Use name_index.csv to find candidate SEID foreign keys and their designation types, and check those corresponding csv and keys further for matching validity/confidence etc...
    - SEID can be numeric and datasets can be sorted by SEID to allow for binary search?

3. Otherwise, Designation Type and Name is not known (e.g. only have an address)
    - Look at the csv headers for clues about the designation type (e.g. only ships have IMO numbers, only individuals have Passport numbers, etc...)
    - Otherwise if limited matching criteria to go off of, scan for matches across all 3 datasets (worst case scenario)

Note: name_index could be expanded with more fields, but there aren't other useful shared fields between all three entity types


## Transform

Create the first instances of each dataset in our database (clean and deduplicate them later)

In [59]:
# Remove unused/unneeded columns
not_used = [
    "OFSI Group ID",
    "UN Reference Number",
    "Designation source",
    "Hull identification number (HIN)"
]
raw_sample.drop(not_used, axis=1, inplace=True)

### 1. name_index.csv

Observations:\
    - Lots of NaNs from later Name fields (longer names less likely)\
    - Some individuals have their full name written in name 1, rather than formatting the surname in name 6\
    - Some people only have a non-latin script name, and vice versa\
    - Varying case script, snakescript etc...\
    - 3 rare entity records had a name 1 field too, not just name 6

i. Create an SEID for each entry in the full dataset

In [60]:
raw_sample["SEID"] = [
    f"#{str(i).zfill(7)}" for i in range(len(raw_sample))
]

ii. Create a Full Name field from merging names 1-6 in order\
The matching algorithm can look for substring similarities within one more densely populated names field, rather than 5 sparse ones

In [61]:
raw_sample["Full Name"] = (
    raw_sample[[                              # Ordered Name columns 1-6
        "Name 1",
        "Name 2",
        "Name 3",
        "Name 4",
        "Name 5",
        "Name 6"
    ]]
    .fillna("")                               # Replace NaN with empty string
    .agg(" ".join, axis=1)                    # Joins name fields with space in between (across columns)
    .str.replace(r"\s+", " ", regex=True)     # Cleans up extra whitespace characters (replace them with just 1 space)
    .str.strip()                              # Remove any left/right trailing spaces
)

# Initialise name_index dataframe
name_index = raw_sample[["SEID", "Full Name", "Designation Type"]].copy()
# raw_sample.drop("Full Name", axis=1, inplace=True)

iii. Extract any other names from the records that may be helpful for matching. Keep the same SEIDS to preserve link to stable record in narrower dataset\
Fields of Interest: Name non-latin script, Subsidiaries, Parent Company, Current & Previous owner

In [62]:
# Other fields that may contain useful matchable names
name_sources = [
    "Name non-latin script",
    "Subsidiaries", 
    "Parent company",      
    "Current owner/operator (s)",
    "Previous owner/operator (s)"
]

# Container for extra rows
extra_names = []


for c in name_sources:
    # Keep the column info and the same SEID for consistency
    temp = raw_sample[raw_sample[c].notna()][["SEID", c, "Designation Type"]].copy()

    # Rename, ready to concat series form
    temp = temp.rename(columns={c: "Full Name"})
    extra_names.append(temp)
extra_names = pd.concat(extra_names, ignore_index=True)

# Add the new names to name_index
name_index = pd.concat(
    [name_index, extra_names],
    ignore_index=True
)

iv. Explode any multiple values per cell, using classic delimiters like ; or | or \n that are unlikely to be naturally occuring in text as parts of names

In [63]:
# Split text whenever delimiter appears with 0+ spaces either side (formatting)
name_index["Full Name"] = (
    name_index["Full Name"]
    .str.split(r"\s*[;|\n]\s*", regex=True)   
)

# List to explode across rows
name_index = name_index.explode("Full Name", ignore_index=True)

# Reformatting (if original text was badly spaced)
name_index["Full Name"] = (
    name_index["Full Name"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [64]:
name_index[name_index["SEID"] == "045434"]

,SEID,Full Name,Designation Type


In [65]:
raw_sample[raw_sample["SEID"] == "045434"][["Name 6", "Subsidiaries", "Parent company", "Name non-latin script"]]

,Name 6,Subsidiaries,Parent company,Name non-latin script


In [66]:
# Save and delete for memory
name_index.to_csv("name_index_raw.csv", index=False)
del name_index
gc.collect()

0

### 2. individuals.csv

i. Merge first and middle names into a Given Names field

In [67]:
raw_sample["Given Names"] = (
    raw_sample[[                              # Ordered Name columns 1-6
        "Name 1",
        "Name 2",
        "Name 3",
        "Name 4",
        "Name 5"
    ]]
    .fillna("")                               # Replace NaN with empty string
    .agg(" ".join, axis=1)                    # Joins name fields with space in between (across columns)
    .str.replace(r"\s+", " ", regex=True)     # Cleans up extra whitespace characters (replace them with just 1 space)
    .str.strip()                              # Remove any left/right trailing spaces
)

ii. Merge address 1-6 into an Address Lines field

In [68]:
raw_sample["Address Lines"] = (
    raw_sample[[                              # Ordered Name columns 1-6
        "Address Line 1",
        "Address Line 2",
        "Address Line 3",
        "Address Line 4",
        "Address Line 5",
        "Address Line 6",
    ]]
    .fillna("")                               # Replace NaN with empty string
    .agg("; ".join, axis=1)                   # Joins name fields with ; in between (across columns) (multi value cell)
    .str.replace(r"\s+", " ", regex=True)     # Cleans up extra whitespace characters (replace them with just 1 space)
    .str.strip()                              # Remove any left/right trailing spaces
)

iii. Initialise the individuals.csv dataset

In [69]:
# Initialise the individuals.csv
individuals = raw_sample[raw_sample["Designation Type"] == "Individual"][[
    "SEID",
    "Unique ID",
    "Name 6",
    "Given Names",
    "Name non-latin script",
    "Non-latin script type",
    "Non-latin script language",
    "Name type",
    "Alias strength",
    "D.O.B",
    "Gender",
    "Title",
    "Position",
    "Nationality(/ies)",
    "Country of birth",
    "Town of birth",
    "Address Lines",
    "Address Postal Code",
    "Address Country",
    "Phone number",
    "Website",
    "Email address",
    "National Identifier number",
    "National Identifier additional information",
    "Passport number",
    "Passport additional information"
]].copy()

iv. Rename columns, save and delete for memory

In [70]:
rename_map = {
    "Unique ID": "LUID",
    "Name 6": "Surname",
    "Name non-latin script": "Name (Non-Latin Script)",
    "Non-latin script type": "Non-Latin Script Type",
    "Non-latin script language": "Non-Latin Script Language",
    "Name type": "Name Type",
    "Alias strength": "Alias Strength",
    "Nationality(/ies)": "Nationalities",
    "Country of birth": "Birth Country",
    "Town of birth": "Birth Town",
    "Phone number": "Phone Number",
    "Email address": "Email",
    "National Identifier number": "National Identifier Number",
    "National Identifier additional information": "National Identifier Info",
    "Passport number": "Passport Number",
    "Passport additional information": "Passport Info"
}

individuals = individuals.rename(columns=rename_map)

In [71]:
# Save and delete for memory
individuals.to_csv("individuals_raw.csv", index=False)
del individuals
gc.collect()

0

### 3. entities.csv

i. Initialise the entities.csv dataset

In [72]:
entities = raw_sample[raw_sample["Designation Type"] == "Entity"][[
    "SEID",
    "Unique ID",
    "Full Name",
    "Name non-latin script",
    "Non-latin script type",
    "Non-latin script language",
    "Name type",
    "Alias strength",
    "Address Lines",
    "Address Postal Code",
    "Address Country",
    "Phone number",
    "Website",
    "Email address",
    "Business registration number (s)",
    "Type of entity",
    "Subsidiaries",
    "Parent company"
]].copy()

ii. Rename columns, save and delete for memory

In [73]:
rename_map = {
    "Unique ID": "LUID",
    "Name non-latin script": "Name (Non-Latin Script)",
    "Non-latin script type": "Non-Latin Script Type",
    "Non-latin script language": "Non-Latin Script Language",
    "Name type": "Name Type",
    "Alias strength": "Alias Strength",
    "Phone number": "Phone Number",
    "Email address": "Email",
    "Business registration number (s)": "Business Reg",
    "Type of entity": "Type",
    "Parent company": "Parent Company"
}

entities = entities.rename(columns=rename_map)

In [74]:
#entities.info()

In [75]:
#raw_sample["Designation Type"].value_counts()

In [76]:
# Save and delete for memory
entities.to_csv("entities_raw.csv", index=False)
del entities
gc.collect()

0

### 4. ships.csv

i. Initialise the ships.csv dataset

In [77]:
ships = raw_sample[raw_sample["Designation Type"] == "Ship"][[
    "SEID",
    "Unique ID",
    "Full Name",
    "Name non-latin script",
    "Non-latin script type",
    "Non-latin script language",
    "Name type",
    "Alias strength",
    "IMO number",
    "Current believed flag of ship",
    "Previous flags",
    "Type of ship",
    "Tonnage of ship",
    "Length of ship",
    "Year Built"
]].copy()

ii. Rename columns, save and delete for memory

In [78]:
rename_map = {
    "Unique ID": "LUID",
    "Name non-latin script": "Name (Non-Latin Script)",
    "Non-latin script type": "Non-Latin Script Type",
    "Non-latin script language": "Non-Latin Script Language",
    "Name type": "Name Type",
    "Alias strength": "Alias Strength",
    "IMO number": "IMO Number",
    "Current believed flag of ship": "Current Believed Flag",
    "Previous flags": "Previous Flags",
    "Type of ship": "Type",
    "Tonnage of ship": "Tonnage",
    "Length of ship": "Length"
}

ships = ships.rename(columns=rename_map)

In [79]:
#ships.info()

In [80]:
#raw_sample["Designation Type"].value_counts()

In [81]:
# Save and delete for memory
ships.to_csv("ships_raw.csv", index=False)
del ships
gc.collect()

0

### 5. sanctions.csv

i. Initialise the sanctions.csv dataset

In [82]:
sanctions = raw_sample[[
    "SEID",
    "Regime Name",
    "Sanctions Imposed",
    "Other Information",
    "UK Statement of Reasons",
    "Date Designated",
    "Last Updated"
]].copy()

ii. Save and delete for memory

In [83]:
#sanctions.info()

In [84]:
#raw_sample["Designation Type"].value_counts()

In [85]:
# Save and delete for memory
sanctions.to_csv("sanctions_raw.csv", index=False)
del sanctions
gc.collect()

0

## Cleaning

Clean each field in each of our datasets

In [111]:
seid_removals = []

### 1. name_index.csv

i. Load raw data

In [86]:
name_index = pd.read_csv("name_index_raw.csv")
name_index.info()

<class 'pandas.DataFrame'>
RangeIndex: 79798 entries, 0 to 79797
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   SEID              79798 non-null  str  
 1   Full Name         79465 non-null  str  
 2   Designation Type  79798 non-null  str  
dtypes: str(3)
memory usage: 1.8 MB


ii. Process each field

- SEID = No Processing Required

- Full Name:
    - Some Missing values can be dropped if there are other name records with the same SEID

In [87]:
mssg_seid = name_index[name_index["Full Name"].isna()]["SEID"].drop_duplicates()
removable = name_index[
    (name_index["SEID"].isin(mssg_seid)) &
    (name_index["Full Name"].notna())
]["SEID"].drop_duplicates()

In [88]:
name_index[name_index["SEID"] == "#0022537"]

,SEID,Full Name,Designation Type
22537,#0022537,Myanmar Economic Corporation,Entity
64944,#0022537,Dagon FC Company Ltd.,Entity
64945,#0022537,NaN,Entity


In [89]:
name_index = name_index[~(
    (name_index["SEID"].isin(removable)) &
    (name_index["Full Name"].isna())
)]
del mssg_seid
del removable
gc.collect()

0

In [94]:
still_mssg = name_index[name_index["Full Name"].isna()]["SEID"].drop_duplicates()

In [96]:
raw_sample.columns

Index(['Last Updated', 'Unique ID', 'Name 6', 'Name 1', 'Name 2', 'Name 3',
       'Name 4', 'Name 5', 'Name type', 'Alias strength', 'Title',
       'Name non-latin script', 'Non-latin script type',
       'Non-latin script language', 'Regime Name', 'Designation Type',
       'Sanctions Imposed', 'Other Information', 'UK Statement of Reasons',
       'Address Line 1', 'Address Line 2', 'Address Line 3', 'Address Line 4',
       'Address Line 5', 'Address Line 6', 'Address Postal Code',
       'Address Country', 'Phone number', 'Website', 'Email address',
       'Date Designated', 'D.O.B', 'Nationality(/ies)',
       'National Identifier number',
       'National Identifier additional information', 'Passport number',
       'Passport additional information', 'Position', 'Gender',
       'Town of birth', 'Country of birth', 'Type of entity', 'Subsidiaries',
       'Parent company', 'Business registration number (s)', 'IMO number',
       'Current owner/operator (s)', 'Previous owner/ope

In [ ]:
raw_sample[raw_sample["SEID"].isin(still_mssg)]["Other Information"].iloc[0]
# Haji title from kabul afghanistan, no names, no information apart from:
#"A close associate of Mullah Mohammed Omar (TAi.004). Member of Taliban Supreme Council as at Dec. 2009. Belongs to Baabar tribe. Review pursuant to Security Council resolution 1822 (2008) was concluded on 21 Jul. 2010. INTERPOL-UN Security Council Special Notice web link: https://www.interpol.int/en/How-we-work/ Notices/View-UN-Notices-Individuals click here"
# I think its safe to say we can remove this, but keep track of these so we can remove records by SEID across all datasets too

seid_removals += still_mssg.values.tolist()

In [115]:
name_index = name_index[~(
    name_index["SEID"].isin(seid_removals)
)]

In [155]:
profile_column(name_index, "Full Name", key=["SEID"])

########################################
Column: Full Name
########################################
Total rows:          79465
Data type:           str
Non-missing values:  79465
Missing values:      0
Missing %:           0.0%
Unique values:       19267
Most common value:   Ministry of Defence and Armed Force Logistics (MODAFL)
Most common count:   3938

Top value counts:
--------------------
Full Name
Ministry of Defence and Armed Force Logistics (MODAFL)                   3938
Iran Aviation Industries Organisation (IAIO) (a subsidiary of MODAFL)    1344
Taghtiran                                                                 588
Taghtiran Co.                                                             588
Taghtiran Kashan                                                          588
Name: count, dtype: int64

Random non-missing examples:
--------------------
#0047025                                                                               Vladimir Vladislavovich KOVALENKO
#0017

In [169]:
raw_sample[raw_sample["SEID"] == "#0010721"].iloc[:,21:]

,Address Line 3,Address Line 4,Address Line 5,Address Line 6,Address Postal Code,Address Country,Phone number,Website,Email address,Date Designated,...,Current believed flag of ship,Previous flags,Type of ship,Tonnage of ship,Length of ship,Year Built,SEID,Full Name,Given Names,Address Lines
10721,NaN,NaN,NaN,Caloocan City,NaN,Philippines,NaN,NaN,NaN,04/06/2008,...,NaN,NaN,NaN,NaN,NaN,NaN,#0010721,So,,10th Avenue; ; ; ; ; Caloocan City


In [ ]:
# Some names are short but they could very well be valid aliases, and sometimes valid acronyms, like CP for Cyber Police

# Plenty of duplicates

### 2. individuals.csv

### 3. entities.csv

### 4. ships.csv

### 5. sanctions.csv

In [ ]:
"""
WE ARE GOING TO HAVE TO REMOVE EMPTY NAMES, "", AT THE END IF NO NAME INFO COULD BE PROVIDED FOR THE RECORD
"""


"""
AS YOU GO THROUGH EACH FIELD IN EACH DATASET< DOCUMENT OBSERVATIONS, QUALITY CONTROL!!!!

"""




"""
SEE profiling function for this, shortest options
"""

## Duplicates

Typically, check the clean transformed data for any duplicates

## QC

Quality control, check if theres any missing values, any obvious duplicates, the range/values fields take, etc...

This is where you do a discussion thing again essentially

## Export